Purpose: Set up data to run maSigPro on downsampled data from *Y. gloriosa* pooled by physiological category.<br>
Author: Anna Pardo<br>
Date initiated: Apr. 13, 2026

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
# load counts
counts = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/counts/Yg_toYgIS_all_md_countsmatrix_over1mil.txt",
                    sep="\t",header="infer")
counts.head()

/tmp/ipykernel_15887/168404881.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  counts = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/counts/Yg_toYgIS_all_md_countsmatrix_over1mil.txt",


,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g,Total_Reads
0,Y1,18,1.0,W,1.0,gloriosa,745,50,0,536,...,42,26,14,196,10,34,173,0,0,12458673
1,Y10,2AB,1.5,W,3.0,gloriosa,176,8,0,93,...,3,57,12,35,0,8,63,0,0,2479905
2,Y100,2AB,6.5,W,23.0,gloriosa,472,26,0,246,...,7,179,41,80,12,17,297,0,3,6344543
3,SRR10848707,55,4.0,D,13.0,gloriosa,523,70,0,232,...,22,404,41,70,142,29,84,0,3,10585018
4,SRR10848820,70,4.0,D,13.0,gloriosa,943,136,0,668,...,81,65,59,215,144,100,298,0,0,21651248


In [3]:
counts["genotype"] = counts["genotype"].astype(str)
counts["genotype"].unique()

array(['18', '2AB', '55', '70', '61', '51', '1AB', '19', '15', 'Eudy',
       'G', '56', '36', '13', '13.0', '18.0', '45.0', '45', '52', '46',
       '43', '37', '53', '48', '51.0', '36.0', '16', '6', '12', '20',
       '50'], dtype=object)

In [4]:
gtmap = {}
for g in counts["genotype"].unique():
    if g.endswith(".0"):
        gtmap[g] = g.rstrip(".0")
    else:
        gtmap[g] = g

In [5]:
counts["genotype"] = counts["genotype"].map(gtmap)
counts["genotype"].unique()

array(['18', '2AB', '55', '70', '61', '51', '1AB', '19', '15', 'Eudy',
       'G', '56', '36', '13', '45', '52', '46', '43', '37', '53', '48',
       '16', '6', '12', '20', '50'], dtype=object)

In [6]:
# load physiology info
phys = json.load(open("/home/leviathan22/Yucca_genomics/phys_figures/physiological_CAM_categories.json"))

In [7]:
# make physiology column
counts["phys"] = counts["genotype"].map(phys)

In [8]:
counts["phys"].unique()

array(['C3', 'possible_fac_CAM', 'facultative_CAM', nan], dtype=object)

In [9]:
def getrepsphys(p,nreps,df=counts):
    pdf = df[df["phys"]==p]
    dfl = []
    for t in pdf["treat"].unique():
        sdf = pdf[pdf["treat"]==t]
        for z in [1.0,5.0,9.0,13.0,17.0,21.0]:
            if z in list(sdf["ZT"]):
                sdf2 = sdf[sdf["ZT"]==z]
                dfl.append(sdf2.sample(n=nreps,axis=0))
    randreps = pd.concat(dfl)
    return randreps

In [10]:
physvals = list(set(list(phys.values())))

In [29]:
dfdict = {}
for i in physvals:
    dfdict[i] = getrepsphys(i,4)

In [30]:
len(dfdict["C3"].index)

48

In [14]:
dfdict["C3"].head()

,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g,Total_Reads,phys
210,Y276,Eudy,1.0,W,1.0,gloriosa,1629,84,1,643,...,48,49,267,18,53,308,0,0,18312461,C3
229,Y3,18,1.0,W,1.0,gloriosa,445,42,1,223,...,16,11,79,2,11,96,0,1,6028428,C3
280,Y375,56,1.0,W,1.0,gloriosa,338,21,1,213,...,207,23,30,3,15,117,0,1,5597203,C3
657,SRR10848765,61,1.0,W,1.0,gloriosa,726,158,0,1202,...,733,57,253,29,50,416,0,2,26684143,C3
284,Y379,56,2.0,W,5.0,gloriosa,521,43,0,315,...,326,17,90,45,24,133,0,5,9850693,C3


In [12]:
r3dict = {}
for i in physvals:
    r3dict[i] = getrepsphys(i,3)

In [16]:
# load Yf counts & subset to 3 reps (has more than 3 reps I think)
yfcounts = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/counts/YfH1_counts_withmd.txt",
                      sep="\t",header="infer")
yfcounts.head()

,sample_name,genotype,time,treat,ZT,species,YufilH1000001m.g,YufilH1000003m.g,YufilH1000007m.g,YufilH1000011m.g,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,X73,2.0,4.0,D,13.0,filamentosa,0,1362,167,327,...,0,45,1051,83,262,243,117,333,0,0
1,X74,2.0,5.0,D,17.0,filamentosa,0,1479,158,293,...,0,30,875,85,256,156,89,334,0,4
2,X64,2.0,2.0,W,5.0,filamentosa,0,380,37,177,...,0,30,694,35,186,94,39,78,0,0
3,X101,27.0,3.0,D,9.0,filamentosa,0,620,141,273,...,0,42,693,62,176,210,55,96,0,2
4,X102,27.0,1.0,D,1.0,filamentosa,0,723,135,200,...,2,23,462,33,189,36,25,358,0,0


In [18]:
nreps = yfcounts.groupby(["treat","ZT"]).count().reset_index()[["treat","ZT","sample_name"]]
nreps.head()

,treat,ZT,sample_name
0,D,1.0,4
1,D,5.0,4
2,D,9.0,4
3,D,13.0,4
4,D,17.0,4


In [19]:
nreps["sample_name"].unique()

array([4, 3])

In [20]:
nreps[nreps["sample_name"]==3]

,treat,ZT,sample_name
6,W,1.0,3


In [22]:
# subset yfcount to 3 reps
dfl = []
for t in yfcounts["treat"].unique():
    sdf = yfcounts[yfcounts["treat"]==t]
    for z in [1.0,5.0,9.0,13.0,17.0,21.0]:
        if z in list(sdf["ZT"]):
            sdf2 = sdf[sdf["ZT"]==z]
            dfl.append(sdf2.sample(n=3,axis=0))
randreps = pd.concat(dfl)

In [23]:
randreps.head()

,sample_name,genotype,time,treat,ZT,species,YufilH1000001m.g,YufilH1000003m.g,YufilH1000007m.g,YufilH1000011m.g,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
15,X118,9.0,1.0,D,1.0,filamentosa,0,526,63,185,...,0,10,840,105,154,13,63,71,0,4
4,X102,27.0,1.0,D,1.0,filamentosa,0,723,135,200,...,2,23,462,33,189,36,25,358,0,0
33,X4,NaN,1.0,D,1.0,filamentosa,0,700,140,382,...,0,74,1011,100,378,43,55,120,0,1
35,X10,NaN,2.0,D,5.0,filamentosa,0,626,150,281,...,0,75,696,66,346,108,52,160,0,0
45,X149,9.0,2.0,D,5.0,filamentosa,1,943,223,420,...,1,30,1606,189,296,229,90,207,0,3


In [24]:
# save this
randreps.to_csv("./inputdata_3reps/YfH1_counts_withmd_3reps_random.txt",sep="\t",header=True,index=False)

In [13]:
# save the other subset data
for k,v in r3dict.items():
    v.to_csv("./inputdata_3reps/Yg_"+k+"_counts_withmd_3reps_random.txt",sep="\t",header=True,index=False)

In [31]:
for k,v in dfdict.items():
    v.to_csv("./inputdata_4reps/Yg_"+k+"_counts_withmd_4reps_random.txt",sep="\t",header=True,index=False)